# Supervised experiment — ML owner

Run the setup cell first. It locates the full repository and adds its src directory to the notebook import path. Set REPO_ROOT_OVERRIDE when the repository is outside the notebook working directory, especially on Databricks serverless. Importing the .ipynb alone does not bring its shared Python modules or dataset.

The initial ingestion/split/candidate cells are runnable once ML dependencies are available. Evaluation and logging remain owner TODOs; follow PRD §7–8 and docs/databricks.md.

In [ ]:
# Locate the full repository before importing its src-layout Python package.
# Local VS Code/Jupyter: leave this empty when running inside the repository.
# Databricks/serverless or another working directory: set the actual absolute
# repository folder containing pyproject.toml, src/, and data/ (not this notebook).
from pathlib import Path
import os
import sys

REPO_ROOT_OVERRIDE = ""  # Example: "/Workspace/Users/<you>/H.A.K-Industry-Hackathon"
configured_root = REPO_ROOT_OVERRIDE or os.environ.get("HF_REPO_ROOT", "")

if configured_root:
    REPO_ROOT = Path(configured_root).expanduser().resolve()
else:
    working_dir = Path.cwd().resolve()
    REPO_ROOT = next(
        (candidate for candidate in (working_dir, *working_dir.parents)
         if (candidate / "pyproject.toml").is_file()
         and (candidate / "src/hf_followup/__init__.py").is_file()),
        None,
    )

if REPO_ROOT is None or not (REPO_ROOT / "src/hf_followup/__init__.py").is_file():
    raise FileNotFoundError(
        "The full repository was not found. Set REPO_ROOT_OVERRIDE to the folder "
        "containing src/hf_followup and data. Importing only this .ipynb into "
        "Databricks does not include the shared Python source; clone/upload the full repo."
    )

SOURCE_DIR = REPO_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

import hf_followup
SOURCE_CSV = REPO_ROOT / "data/heart_failure_clinical_records.csv"
if not SOURCE_CSV.is_file():
    raise FileNotFoundError(f"Bundled dataset missing: {SOURCE_CSV}. Include the repo's data folder.")
print(f"Package imported from: {hf_followup.__file__}")
print(f"Dataset: {SOURCE_CSV}")
print(f"Notebook Python: {sys.executable}")


In [ ]:
# ML dependencies must be installed in THIS notebook's Python environment.
# If pandas/numpy/scikit-learn are missing locally, run in a separate notebook cell:
# %pip install -e "{REPO_ROOT}[ml]"
# Then restart the kernel and rerun from cell 2 if your environment requires it.
# In Databricks, first check the compute/environment's installed packages; install
# compatible ML dependencies through its supported environment settings if needed.
from hf_followup.data.ingest import ingest_csv
from hf_followup.ml.training import (
    training_frame, fixed_split, candidate_pipelines, development_folds, ALPHA,
)

# These initial data/split steps are ready to run; no model fitting happens here.
ingestion = ingest_csv(SOURCE_CSV)
x, y = training_frame(ingestion.cohort, ingestion.outcomes)
split = fixed_split(ingestion.cohort, ingestion.outcomes)
development_ids, test_ids = split["development_ids"], split["test_ids"]
assert len(development_ids) == 239 and len(test_ids) == 60
assert not set(development_ids) & set(test_ids)
print(f"Accepted records: {len(x)}; development: {len(development_ids)}; test: {len(test_ids)}")
# TODO(ML): save split_manifest.json with source hash/software versions before tuning.


In [ ]:
# Constructors fit no data. Scaling stays inside each fold-local pipeline.
pipelines = candidate_pipelines()
folds = development_folds()
print({name: str(pipeline) for name, pipeline in pipelines.items()})
# TODO: declare compact hyperparameter grid and practical-equivalence selection rule
# before results. x never contains time/DEATH_EVENT.

## TODO: development CV and selection

Evaluate age/points references, logistic regression and random forest on identical five validation folds. Clone/refit full pipelines on fold training only. K_eval = ceil((25/299) × validation N). Save precision@K, capture@K, lift, ROC-AUC, average precision, Brier (probabilities), per-fold values and variability. Select by mean capacity capture/precision, then discrimination/stability; prefer the simpler explainable pipeline when practically indistinguishable. Log every candidate to Databricks MLflow. Never use the held-out test to select/tune.

In [ ]:
# TODO: implement CV here using sklearn.base.clone and fold-local fit.
# for train_idx, val_idx in folds.split(x.loc[development_ids], y.loc[development_ids]):
#     train_ids = [development_ids[i] for i in train_idx]
#     validation_ids = [development_ids[i] for i in val_idx]
#     fitted = clone(candidate).fit(x.loc[train_ids], y.loc[train_ids])
#     scores = fitted.predict_proba(x.loc[validation_ids])[:, 1]
#     ... capacity metrics with shared hash tie policy ...
#     ... age/points references on the same validation IDs ...
# Persist fold IDs/results, parameters, seed, versioned selection rationale.

## TODO: frozen test and publication

Refit selected pipeline on development only. Freeze; evaluate once on 60 untouched test records with K=6. Log complete pipeline, signature/input example, immutable run URI, split, metadata, reports and compatible versions. Produce model_metadata.json, split_manifest.json, evaluation_report.json and trusted offline artifact if needed; verify reload scores/order. Publish Delta predictions tagged development_in_sample/held_out_test. Logistic contribution = fitted-scaled value × coefficient in log-odds plus intercept. Forest P0 = recorded indicators; local attribution unavailable. No global importances passed as local attribution, or uncalibrated clinical percentages. Report disappointing results honestly.